# Cleaning Pipeline 04 — Duplicate and Semantic Similarity with DINOv3 Embeddings

This notebook converts each valid image into a normalized DINOv3 feature vector and searches for nearest neighbors with cosine similarity.

It can surface resized/recompressed copies, crops, repeated scenes, semantically similar images, and possible cross-label conflicts.

**Important:** high embedding similarity does not prove duplication. Different valid examples of the same object can be close in embedding space. Visual review is mandatory.

In [ ]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "cleaning_pipeline":
    REPO_ROOT = REPO_ROOT.parents[1]
elif REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent

sys.path.insert(0, str(REPO_ROOT / "src"))
DATA_ROOT = REPO_ROOT / "BDC2026"
OUTPUT_ROOT = REPO_ROOT / "eda_outputs" / "notebook_pipeline"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
LABEL2ID = {"0_Recyclable": 0, "1_Electronic": 1, "2_Organic": 2}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}
print("REPO_ROOT:", REPO_ROOT)
print("DATA_ROOT:", DATA_ROOT)
print("OUTPUT_ROOT:", OUTPUT_ROOT)

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from PIL import Image, ImageOps
from sklearn.neighbors import NearestNeighbors
from tqdm.auto import tqdm
from transformers import AutoImageProcessor, AutoModel
from dotenv import load_dotenv

load_dotenv(REPO_ROOT / ".env")

MODEL_NAME = "facebook/dinov3-vitl16-pretrain-lvd1689m"
HF_TOKEN = os.environ.get("HF_TOKEN")
BATCH_SIZE = 16
N_NEIGHBORS = 8
SIMILARITY_THRESHOLD = 0.985

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

In [ ]:
manifest_path = OUTPUT_ROOT / "00_manifest" / "train_manifest.csv"
if not manifest_path.exists():
    raise FileNotFoundError(f"Run notebook 00 first: {manifest_path}")
manifest = pd.read_csv(manifest_path)
valid = manifest[manifest["is_valid"]].copy().reset_index(drop=True)
print("Valid images:", len(valid))

In [ ]:
processor = AutoImageProcessor.from_pretrained(MODEL_NAME, token=HF_TOKEN)
model = AutoModel.from_pretrained(MODEL_NAME, token=HF_TOKEN).to(device)
model.eval()

all_embeddings = []
kept_rows = []

with torch.no_grad():
    for start in tqdm(range(0, len(valid), BATCH_SIZE), desc="DINO embeddings"):
        batch_df = valid.iloc[start:start + BATCH_SIZE]
        images, batch_rows = [], []
        for _, row in batch_df.iterrows():
            try:
                with Image.open(row["path"]) as img:
                    images.append(ImageOps.exif_transpose(img).convert("RGB"))
                batch_rows.append(row.to_dict())
            except Exception as exc:
                print("Skipped:", row["path"], repr(exc))
        if not images:
            continue

        inputs = processor(images=images, return_tensors="pt").to(device)
        outputs = model(**inputs)
        features = outputs.pooler_output if getattr(outputs, "pooler_output", None) is not None else outputs.last_hidden_state[:, 0]
        features = torch.nn.functional.normalize(features.float(), p=2, dim=1)
        all_embeddings.append(features.cpu().numpy())
        kept_rows.extend(batch_rows)

embeddings = np.concatenate(all_embeddings, axis=0)
embedding_index = pd.DataFrame(kept_rows).reset_index(drop=True)
assert len(embeddings) == len(embedding_index)
print("Embedding matrix:", embeddings.shape)

In [ ]:
stage_dir = OUTPUT_ROOT / "04_dino_embeddings"
stage_dir.mkdir(parents=True, exist_ok=True)
np.save(stage_dir / "dino_embeddings.npy", embeddings)
embedding_index[["path","relative_path","class_name","label"]].to_csv(
    stage_dir / "dino_embedding_index.csv", index=False
)
print("Saved embeddings and index.")

In [ ]:
neighbors = NearestNeighbors(
    n_neighbors=min(N_NEIGHBORS + 1, len(embedding_index)),
    metric="cosine",
    algorithm="brute",
    n_jobs=-1,
)
neighbors.fit(embeddings)
distances, indices = neighbors.kneighbors(embeddings)

pairs, seen = [], set()
for i in tqdm(range(len(embedding_index)), desc="Building similarity pairs"):
    for distance, j in zip(distances[i], indices[i]):
        if i == j:
            continue
        similarity = 1.0 - float(distance)
        if similarity < SIMILARITY_THRESHOLD:
            continue
        key = (min(i,j), max(i,j))
        if key in seen:
            continue
        seen.add(key)
        row_i, row_j = embedding_index.iloc[i], embedding_index.iloc[j]
        pairs.append({
            "path_a": row_i["path"],
            "relative_path_a": row_i["relative_path"],
            "class_a": row_i["class_name"],
            "label_a": int(row_i["label"]),
            "path_b": row_j["path"],
            "relative_path_b": row_j["relative_path"],
            "class_b": row_j["class_name"],
            "label_b": int(row_j["label"]),
            "cosine_similarity": similarity,
            "cross_label": int(row_i["label"]) != int(row_j["label"]),
        })

pair_columns = ["path_a", "relative_path_a", "class_a", "label_a", "path_b", "relative_path_b", "class_b", "label_b", "cosine_similarity", "cross_label"]
dino_pairs = pd.DataFrame(pairs, columns=pair_columns)
if len(dino_pairs):
    dino_pairs = dino_pairs.sort_values(["cross_label","cosine_similarity"], ascending=[False,False])
print("Pairs above threshold:", len(dino_pairs))
print("Cross-label pairs:", int(dino_pairs["cross_label"].sum()) if len(dino_pairs) else 0)
display(dino_pairs.head(30))

In [ ]:
nearest_nonself_similarity = 1.0 - distances[:, 1]
plt.figure(figsize=(8,4))
plt.hist(nearest_nonself_similarity, bins=100)
plt.axvline(SIMILARITY_THRESHOLD, linestyle="--", label=f"threshold={SIMILARITY_THRESHOLD}")
plt.xlabel("Nearest non-self cosine similarity")
plt.ylabel("Images")
plt.title("Nearest-neighbor similarity distribution")
plt.legend(); plt.show()

display(pd.Series(nearest_nonself_similarity).describe(percentiles=[.90,.95,.99,.995,.999]))

In [ ]:
def show_pairs(df, n=12, title="DINO similarity candidates"):
    sample = df.head(n)
    if len(sample) == 0:
        print("No pairs to display"); return
    fig, axes = plt.subplots(len(sample), 2, figsize=(8, len(sample)*3.0))
    axes = np.asarray(axes).reshape(len(sample),2)
    for row_idx, (_, row) in enumerate(sample.iterrows()):
        for col_idx, side in enumerate(["a","b"]):
            path = row[f"path_{side}"]
            with Image.open(path) as img:
                axes[row_idx,col_idx].imshow(ImageOps.exif_transpose(img).convert("RGB"))
            axes[row_idx,col_idx].axis("off")
            axes[row_idx,col_idx].set_title(
                f"{Path(path).name} | {row[f'class_{side}']}\n"
                f"sim={row['cosine_similarity']:.5f} cross_label={row['cross_label']}", fontsize=8)
    fig.suptitle(title); plt.tight_layout(); plt.show()

cross = dino_pairs[dino_pairs["cross_label"]] if len(dino_pairs) else dino_pairs
show_pairs(cross if len(cross) else dino_pairs, title="DINO similarity review")

## Suggested review labels

- `same_file_or_transform`
- `same_scene_different_frame`
- `same_object_different_photo`
- `different_objects_same_visual_pattern`
- `cross_label_possible_annotation_error`
- `ambiguous_class_boundary`

Only the first two categories are realistic removal candidates, and even then preserve enough diversity.

In [ ]:
dino_pairs.to_csv(stage_dir / "dino_duplicate_pairs.csv", index=False)
review = dino_pairs.copy()
review["review_decision"] = ""
review["review_notes"] = ""
review.to_csv(stage_dir / "dino_pairs_for_manual_review.csv", index=False)
print("Saved outputs to", stage_dir)

del model
if torch.cuda.is_available():
    torch.cuda.empty_cache()